# ChargebackOps Merchant Agent - SFT + GRPO on Qwen2.5 3B (fp16 LoRA)

Two-phase training pipeline for a Kaggle/Colab T4:

1. **Phase A - SFT** on heuristic rollouts. Teaches the base model the JSON action schema and per-state action variety.
2. **Phase B - GRPO** from the SFT checkpoint. Per-action reward against the heuristic oracle (parse-fail = 0, unavailable = 0.1, wrong action_type = 0.4, wrong target = 0.7, exact = 1.0).
3. **Eval** every checkpoint (overall + per-difficulty curves) using rollouts with `_predicted_noop` stall detection so the rollout reaches grading even if the model loops.

**Default model:** `Qwen/Qwen2.5-3B-Instruct`. This matches the hackathon guide's Qwen2.5 GRPO recipe family and is public enough for Kaggle without the Gemma license gate. If T4 memory is too tight, use `MODEL_ID=Qwen/Qwen2.5-0.5B-Instruct` only as a smoke-test fallback.

**Why Qwen2.5 3B now**: the hackathon guide explicitly points teams at Qwen2.5 3B GRPO-style recipes. Gemma is blocked by gating/OOM in this runtime, and Qwen2.5 3B is the closest authority-aligned trainable option on Kaggle.

**Runtime requirement:** Kaggle/Colab T4 GPU with internet enabled. Verify with `nvidia-smi` in cell 0.


## 0. Setup — install deps + clone repo

In [ ]:
# GPU + repo setup. Kaggle/Colab T4. Verify GPU before running.
# Fix stale cwd FIRST (a prior failed cell may have deleted it).
import os
os.chdir('/content' if os.path.isdir('/content') else '/kaggle/working')
WORK_DIR = os.getcwd()

import shutil, subprocess
print(subprocess.check_output(['nvidia-smi', '-L']).decode())

# STEP 1 - pin torch trio to the matched cu128 set. Prior installs (or Colab
# pre-installs) sometimes pull in torch 2.11+cu130 which is incompatible
# with torchvision 0.25+cu128 and breaks every downstream import.
subprocess.run(
    ['pip', 'install', '-q', '--no-cache-dir',
     'torch==2.10.0', 'torchvision==0.25.0', 'torchaudio==2.10.0',
     '--index-url', 'https://download.pytorch.org/whl/cu128'],
    check=True, cwd=WORK_DIR,
)

# STEP 2a - FORCE exact pins. Keep the training stack fixed so Kaggle/Colab
# preinstalls do not silently change Transformers, TRL, PEFT, or HF Hub behavior.
# `--no-deps` keeps torch untouched.
subprocess.run(
    ['pip', 'install', '-q', '--no-cache-dir', '--force-reinstall', '--no-deps',
     'transformers==5.5.4', 'trl==0.20.0', 'peft==0.14.0',
     'tokenizers==0.22.2', 'huggingface-hub==1.11.0'],
    check=True, cwd=WORK_DIR,
)

# STEP 2b - supporting libs with only-if-needed so torch stays put.
subprocess.run(
    ['pip', 'install', '-q', '--upgrade-strategy=only-if-needed',
     'accelerate>=0.30,<2.0', 'datasets>=2.20,<4.0',
     'matplotlib>=3.8', 'pydantic>=2.10',
     'openenv-core>=0.2.2'],
    check=True, cwd=WORK_DIR,
)

# Clone repo (always fresh).
REPO_DIR = os.path.join(WORK_DIR, 'chargebackops')
if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)
subprocess.run(
    ['git', 'clone', '--depth', '1',
     'https://github.com/MitudruDutta/ChargeBackOps.git', REPO_DIR],
    check=True, cwd=WORK_DIR,
)
os.chdir(REPO_DIR)
print('cwd:', os.getcwd())

# Editable install with --no-deps so pyproject's app/server deps don't alter
# the training stack we just pinned.
subprocess.run(['pip', 'install', '-q', '-e', '.', '--no-deps'],
               check=True, cwd=REPO_DIR)

# Verify all critical pins land at the exact requested versions.
import importlib.metadata as md
print('torch        ', md.version('torch'))
print('torchvision  ', md.version('torchvision'))
print('transformers ', md.version('transformers'), '(want 5.5.4)')
print('tokenizers   ', md.version('tokenizers'),   '(want 0.22.2)')
print('hf-hub       ', md.version('huggingface-hub'), '(want 1.11.0)')
print('trl          ', md.version('trl'),          '(want 0.20.0)')
print('peft         ', md.version('peft'),         '(want 0.14.0)')
print('accelerate   ', md.version('accelerate'))
print('openenv-core ', md.version('openenv-core'))
assert md.version('transformers') == '5.5.4', 'transformers pin failed'
assert md.version('trl') == '0.20.0', 'trl pin failed'
assert md.version('peft') == '0.14.0', 'peft pin failed'
assert md.version('tokenizers') == '0.22.2', 'tokenizers pin failed'
assert md.version('huggingface-hub') == '1.11.0', 'huggingface-hub pin failed'


In [ ]:
# Path + module-cache flush so the editable install resolves before any other import.
import os, sys, importlib, logging, torch
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# Silence transformers per-layer "Caching is incompatible..." spam at scale.
import transformers
transformers.logging.set_verbosity_error()
logging.getLogger('transformers').setLevel(logging.ERROR)
for name in [
    'transformers.models.qwen2.modeling_qwen2',
    'transformers.models.gemma4.modeling_gemma4',
]:
    logging.getLogger(name).setLevel(logging.ERROR)

REPO_DIR = '/content/chargebackops' if os.path.isdir('/content/chargebackops') else '/kaggle/working/chargebackops'
sys.path.insert(0, REPO_DIR)
importlib.invalidate_caches()
for mod in list(sys.modules):
    if mod.startswith(('scenarios', 'training', 'evaluation', 'server', 'core', 'runners', 'connectors')):
        del sys.modules[mod]
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('repo:', REPO_DIR)


## 1. Load Qwen2.5 3B-Instruct in fp16 + attach LoRA adapter

* **fp16 base** - no `bitsandbytes`, no quantization wheel mismatch; Qwen2.5 3B is the authority-aligned default for Kaggle T4.
* **Chat template** - uses the selected model chat format so SFT and inference use the same prompt surface.
* **Override model** - set `MODEL_ID` in the environment before running the cell if you need a fallback.


In [ ]:
from transformers import AutoModelForCausalLM, AutoProcessor, AutoTokenizer
from peft import LoraConfig, get_peft_model

MODEL_ID = os.environ.get('MODEL_ID', 'Qwen/Qwen2.5-3B-Instruct')
IS_GEMMA4 = 'gemma-4' in MODEL_ID.lower()
print('MODEL_ID:', MODEL_ID)

# Gemma 4 uses AutoProcessor; most public text-only models use AutoTokenizer.
if IS_GEMMA4:
    processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
    tokenizer = getattr(processor, 'tokenizer', processor)
else:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    processor = tokenizer
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'  # required for GRPO generation

def render_chat(messages, *, add_generation_prompt: bool) -> str:
    kwargs = {
        'tokenize': False,
        'add_generation_prompt': add_generation_prompt,
    }
    if IS_GEMMA4:
        kwargs['enable_thinking'] = False
    try:
        return processor.apply_chat_template(messages, **kwargs)
    except TypeError:
        kwargs.pop('enable_thinking', None)
        return processor.apply_chat_template(messages, **kwargs)

def encode_text(text: str, **kwargs):
    try:
        return processor(text=text, **kwargs)
    except TypeError:
        return tokenizer(text, **kwargs)

def decode_tokens(token_ids, *, skip_special_tokens: bool = False) -> str:
    decoder = processor if hasattr(processor, 'decode') else tokenizer
    return decoder.decode(token_ids, skip_special_tokens=skip_special_tokens)

# T4 = Turing (sm_75), no bf16 hardware. fp16 only.
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
)
base_model.gradient_checkpointing_enable()
base_model.enable_input_require_grads()  # required for LoRA + grad checkpoint

# Gemma 4 wraps projections as Gemma4ClippableLinear(linear=nn.Linear).
# PEFT 0.14 cannot LoRA-wrap the wrapper, so target the inner .linear modules.
# Public text-only defaults use normal projection names and the standard target list.
if IS_GEMMA4:
    lora_target_modules = r'.*(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)\.linear$'
    lora_rank = 8
    lora_alpha = 16
else:
    lora_target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                           'gate_proj', 'up_proj', 'down_proj']
    lora_rank = 16
    lora_alpha = 32

lora_config = LoraConfig(
    r=lora_rank,
    lora_alpha=lora_alpha,
    target_modules=lora_target_modules,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()
print(f'VRAM allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB | '
      f'free: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB')


## 2. Phase A — SFT on heuristic rollouts

Builds (prompt, oracle_completion) pairs by rolling the scripted heuristic on every headline + generated task. Wraps examples in the selected model chat template so the model learns the same prompt format used at inference time.

After this phase the model emits valid JSON with the right `action_type` per state — solves the *always emit `select_case`* collapse before GRPO ever runs.

In [ ]:
from datasets import Dataset
from scenarios.simulation import list_tasks, get_task
from training.sft_dataset import build_sft_dataset

seeds = [7, 17, 31, 42, 53, 77, 99, 113, 131, 157]
task_ids = [t.task_id for t in list_tasks()]
for diff in ['easy', 'medium', 'hard', 'nightmare']:
    for s in seeds:
        tid = f'generated_{diff}_s{s}'
        try:
            get_task(tid)
            if tid not in task_ids:
                task_ids.append(tid)
        except Exception:
            pass

raw_sft = build_sft_dataset(task_ids, max_states_per_task=24)

def to_chat_text(prompt: str, completion: str) -> str:
    return render_chat(
        [
            {'role': 'user', 'content': prompt},
            {'role': 'assistant', 'content': completion},
        ],
        add_generation_prompt=False,
    )

sft_rows = [{'text': to_chat_text(s['prompt'], s['completion'])} for s in raw_sft]
sft_dataset = Dataset.from_list(sft_rows)

from collections import Counter
atype_counts = Counter(s['action_type'] for s in raw_sft)
print(f'SFT samples: {len(sft_dataset)}, unique tasks: {len(set(s["task_id"] for s in raw_sft))}')
print(f'action_type distribution: {dict(atype_counts)}')
print('sample (first 500 chars):')
print(sft_rows[0]['text'][:500])

In [ ]:
from trl import SFTConfig, SFTTrainer

OUT_ROOT = '/content' if os.path.isdir('/content') else '/kaggle/working'
SFT_DIR = os.path.join(OUT_ROOT, 'sft-merchant-agent')
GRPO_DIR = os.path.join(OUT_ROOT, 'grpo-merchant-agent')

sft_config = SFTConfig(
    output_dir=SFT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=2,
    learning_rate=2e-4,
    logging_steps=10,
    save_steps=200,
    save_total_limit=2,
    bf16=False,
    fp16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    max_length=1024,
    dataset_text_field='text',
    report_to='none',
    optim='adamw_torch',
    warmup_ratio=0.05,
)

# Silence the per-layer "Caching is incompatible with gradient checkpointing"
# spam by disabling KV cache up-front (grad checkpoint disables it per forward
# anyway). Same line is set before GRPO for the same reason.
if hasattr(model, 'config'):
    model.config.use_cache = False

sft_trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=sft_dataset,
    processing_class=tokenizer,
)
sft_trainer.train()
sft_trainer.save_model(os.path.join(SFT_DIR, 'final'))
print(f'PEAK VRAM (SFT): {torch.cuda.max_memory_allocated()/1e9:.2f} GB')

## 2.5. Merge SFT LoRA into base, attach fresh LoRA for GRPO

`accelerate.unwrap_model_for_generation` calls `merge_adapter()`+`unmerge_adapter()` around generation. With fp16 LoRA the round-trip loses enough precision that completions degrade to base-model noise — exactly the all-zero-reward GRPO failure mode.

Fix: bake SFT into the base weights via `merge_and_unload()`, then attach a fresh zero-initialized LoRA. The fresh adapter starts as identity, so generation emits SFT-quality output regardless of how TRL toggles adapter state. GRPO then trains the fresh adapter on top.

In [ ]:
# Merge SFT LoRA into base weights, then attach a fresh LoRA for GRPO.
print(f'before merge: VRAM {torch.cuda.memory_allocated()/1e9:.2f} GB')

# merge_and_unload() returns the unwrapped base model with adapter weights
# folded in. The original LoRA wrapper + adapter tensors are freed.
merged_base = model.merge_and_unload()
del model
torch.cuda.empty_cache()
print(f'after merge: VRAM {torch.cuda.memory_allocated()/1e9:.2f} GB')

# Re-arm grad-checkpoint hook on the embedding output (lost across merge).
merged_base.enable_input_require_grads()

# Sanity: SFT-baked base should still emit clean JSON deterministically.
from training.env_adapter import build_prompt
from server.chargeback_ops_environment import ChargebackOpsEnvironment
env = ChargebackOpsEnvironment()
obs = env.reset(task_id='goods_not_received_easy')
chat = render_chat(
    [{'role': 'user', 'content': build_prompt(obs.model_dump())}],
    add_generation_prompt=True,
)
inp = encode_text(chat, return_tensors='pt').to(merged_base.device)
merged_base.eval()
with torch.no_grad():
    out = merged_base.generate(**inp, max_new_tokens=160, do_sample=False,
                               pad_token_id=tokenizer.eos_token_id)
print('merged-base gen:', repr(decode_tokens(out[0][inp.input_ids.shape[1]:], skip_special_tokens=False)))
merged_base.train()

# Attach fresh GRPO LoRA. lora_dropout=0 so generation can't be polluted by
# random adapter contributions even if TRL leaves the model in train() mode.
lora_grpo = LoraConfig(
    r=lora_rank,
    lora_alpha=lora_alpha,
    target_modules=lora_target_modules,
    lora_dropout=0.0,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(merged_base, lora_grpo)
model.enable_input_require_grads()
model.print_trainable_parameters()
print(f'after fresh LoRA: VRAM {torch.cuda.memory_allocated()/1e9:.2f} GB')

## 3. Phase B — GRPO from the SFT checkpoint

Per-action reward vs. heuristic oracle at the dataset's recorded state. With SFT done, the model already emits valid actions — GRPO now sharpens *which* action_type to pick at borderline states.

In [ ]:
from training.reward_adapter import build_state_action_dataset

raw_grpo = build_state_action_dataset(task_ids, max_states_per_task=14)

def to_chat_prompt(prompt: str) -> str:
    return render_chat(
        [{'role': 'user', 'content': prompt}],
        add_generation_prompt=True,
    )

grpo_rows = [
    {
        'prompt': to_chat_prompt(s['prompt']),
        'task_id': s['task_id'],
        'state_step': int(s['state_step']),
    }
    for s in raw_grpo
]
grpo_dataset = Dataset.from_list(grpo_rows)
print(f'GRPO samples: {len(grpo_dataset)}, unique tasks: {len(set(r["task_id"] for r in grpo_rows))}')

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from training.reward_adapter import compute_reward
import torch.nn as nn

def reward_fn(prompts, completions, **kwargs):
    task_ids = kwargs.get('task_id') or kwargs.get('task_ids')
    state_steps = kwargs.get('state_step') or kwargs.get('state_steps')
    return compute_reward(prompts, completions, task_ids=task_ids, state_steps=state_steps)

# Re-arm the require-grad hook (lost across merge_and_unload + get_peft_model).
model.enable_input_require_grads()
if hasattr(model, 'config'):
    model.config.use_cache = False

# Force chat termination into both model config and generation config.
# Without this, TRL GRPO can sample valid-looking JSON but never emit EOS,
# causing every completion to hit max_completion_length and every reward to be 0.
model.config.eos_token_id = tokenizer.eos_token_id
model.config.pad_token_id = tokenizer.pad_token_id
model.generation_config.eos_token_id = tokenizer.eos_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id

# Belt-and-suspenders dropout zeroing in case any new Dropout module slipped in.
for m in model.modules():
    if isinstance(m, nn.Dropout):
        m.p = 0.0

# GRPO must stay close to the SFT policy. A previous temp=0.7 / 192-token run
# clipped every completion, produced zero reward variance, and gave grad_norm=0.
# These settings are intentionally conservative: shorter completions, explicit
# EOS/pad IDs, low-temperature sampling, and completion logging for debugging.
grpo_config = GRPOConfig(
    output_dir=GRPO_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_generations=2,
    max_prompt_length=768,
    max_completion_length=96,
    learning_rate=2e-6,
    max_steps=200,
    logging_steps=5,
    save_steps=50,
    save_total_limit=4,
    bf16=False,
    fp16=True,
    gradient_checkpointing=False,
    report_to='none',
    beta=0.0,
    temperature=0.2,
    top_p=0.8,
    top_k=20,
    repetition_penalty=1.05,
    generation_kwargs={
        'eos_token_id': tokenizer.eos_token_id,
        'pad_token_id': tokenizer.pad_token_id,
    },
    log_completions=True,
    num_completions_to_print=2,
    optim='adamw_torch',
)

grpo_trainer = GRPOTrainer(
    model=model,  # SFT weights baked into base; trainable fresh LoRA on top
    processing_class=tokenizer,
    reward_funcs=[reward_fn],
    args=grpo_config,
    train_dataset=grpo_dataset,
)
grpo_trainer.train()
print(f'PEAK VRAM (GRPO): {torch.cuda.max_memory_allocated()/1e9:.2f} GB')

## 4. Per-checkpoint eval — overall + per-family

Loads each saved adapter checkpoint, plays full episodes across the headline catalog, and plots the curve. Stall detection in `run_episode_with_text_policy` ensures degenerate checkpoints (e.g. early SFT that still loops) reach grading instead of returning 0.

In [ ]:
import glob, re
from peft import PeftModel
from transformers import AutoProcessor, AutoTokenizer
from training.curve import (
    evaluate_checkpoint, evaluate_checkpoint_by_family,
    plot_training_curve, plot_training_curve_by_family,
)

# Eval pipeline: load each adapter into the correct fp16 base for inference.
# GRPO adapters were trained on top of the SFT-merged base, so evaluating
# them on the raw base is invalid. Reconstruct SFT-merged base first.
def make_text_policy(adapter_path: str | None, adapter_kind: str = 'base'):
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, dtype=torch.float16, device_map='auto', trust_remote_code=True,
    )
    if adapter_kind == 'grpo':
        sft_model = PeftModel.from_pretrained(base, os.path.join(SFT_DIR, 'final'))
        base = sft_model.merge_and_unload()
        m = PeftModel.from_pretrained(base, adapter_path)
    elif adapter_path is not None:
        m = PeftModel.from_pretrained(base, adapter_path)
    else:
        m = base
    m.eval()

    if IS_GEMMA4:
        proc = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
        tok = getattr(proc, 'tokenizer', proc)
    else:
        tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
        proc = tok
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = 'left'

    def local_render(messages, *, add_generation_prompt: bool) -> str:
        kwargs = {'tokenize': False, 'add_generation_prompt': add_generation_prompt}
        if IS_GEMMA4:
            kwargs['enable_thinking'] = False
        try:
            return proc.apply_chat_template(messages, **kwargs)
        except TypeError:
            kwargs.pop('enable_thinking', None)
            return proc.apply_chat_template(messages, **kwargs)

    def local_encode(text: str, **kwargs):
        try:
            return proc(text=text, **kwargs)
        except TypeError:
            return tok(text, **kwargs)

    def local_decode(token_ids, *, skip_special_tokens: bool = True) -> str:
        decoder = proc if hasattr(proc, 'decode') else tok
        return decoder.decode(token_ids, skip_special_tokens=skip_special_tokens)

    def policy(prompt: str) -> str:
        chat = local_render(
            [{'role': 'user', 'content': prompt}],
            add_generation_prompt=True,
        )
        inputs = local_encode(chat, return_tensors='pt', truncation=True, max_length=1024).to(m.device)
        with torch.no_grad():
            out = m.generate(
                **inputs, max_new_tokens=256, do_sample=False,
                pad_token_id=tok.eos_token_id,
                eos_token_id=tok.eos_token_id,
            )
        return local_decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return policy, m

# Catalog of checkpoints: untrained base, SFT final, then every GRPO save.
ckpt_specs = [('base', None, 0, 'base')]
ckpt_specs.append(('sft', os.path.join(SFT_DIR, 'final'), 1, 'sft'))
grpo_dirs = sorted(
    glob.glob(os.path.join(GRPO_DIR, 'checkpoint-*')),
    key=lambda p: int(re.search(r'checkpoint-(\d+)', p).group(1)),
)
for d in grpo_dirs:
    step = int(re.search(r'checkpoint-(\d+)', d).group(1))
    ckpt_specs.append((f'grpo-{step}', d, 1 + step, 'grpo'))

overall = []
grouped = []
for label, path, step, kind in ckpt_specs:
    print(f'eval {label} from {path}')
    pol, m_ckpt = make_text_policy(path, kind)
    overall.append(evaluate_checkpoint(step=step, policy=pol))
    grouped.append(evaluate_checkpoint_by_family(step=step, policy=pol))
    del m_ckpt
    torch.cuda.empty_cache()

print('\nOVERALL CURVE:')
for c in overall:
    print(f'  step={c.step:4d} mean={c.mean_score:.4f}')

print('\nPER-FAMILY CURVE:')
for g in grouped:
    line = f'  step={g.step:4d}'
    for fam in sorted(g.by_family.keys()):
        line += f'  {fam}={g.by_family[fam].mean_score:.3f}'
    print(line)

from runners.benchmark_runner import run_policy_sweep
sweep = run_policy_sweep()
heur_overall = next(s.mean_score for s in sweep.policies if s.policy == 'heuristic')

FIG_DIR = os.path.join(REPO_DIR, 'docs', 'figures')
os.makedirs(FIG_DIR, exist_ok=True)
plot_training_curve(
    overall, os.path.join(FIG_DIR, 'training_curve.png'),
    baseline_scores={'heuristic': heur_overall, 'naive': 0.0},
)
plot_training_curve_by_family(
    grouped, os.path.join(FIG_DIR, 'training_curve_by_family.png'),
    family_order=['easy', 'medium', 'hard', 'nightmare'],
)
print(f'\nfigures saved to {FIG_DIR}/')


## 5. Diagnose final checkpoint

Print the final checkpoint's completion vs. the heuristic oracle on three representative tasks (easy, hard, nightmare). Verifies the model emits valid JSON and matches the oracle on the initial state.

In [ ]:
from training.env_adapter import build_prompt, parse_completion
from training.reward_adapter import compute_reward
from server.chargeback_ops_environment import ChargebackOpsEnvironment
from runners.benchmark_runner import heuristic_policy

final_adapter = grpo_dirs[-1] if grpo_dirs else '/content/sft-merchant-agent/final'
print(f'diagnose adapter: {final_adapter}')
final_kind = 'grpo' if grpo_dirs else 'sft'
policy, m = make_text_policy(final_adapter, final_kind)

for tid in ['goods_not_received_easy', 'queue_optimization_hard', 'generated_nightmare_s31']:
    env = ChargebackOpsEnvironment()
    obs = env.reset(task_id=tid)
    raw = build_prompt(obs.model_dump())
    completion = policy(raw)
    parsed = parse_completion(completion)
    oracle = heuristic_policy(obs.model_dump())
    reward = compute_reward(['x'], [completion], task_ids=[tid], state_steps=[0])[0]
    print(f'\n=== {tid} ===')
    print(f'oracle: {oracle.action_type} case={oracle.case_id}')
    print(f'completion (first 200): {repr(completion[:200])}')
    print(f'parsed: {parsed}')
    print(f'reward vs oracle: {reward:.3f}')
del m
torch.cuda.empty_cache()

## Done

Artifacts written to `/content/chargebackops/docs/figures/` or `/kaggle/working/chargebackops/docs/figures/`:
* `training_curve.png` - overall mean score across SFT + GRPO checkpoints, with heuristic baseline.
* `training_curve_by_family.png` - per-difficulty curves (easy / medium / hard / nightmare).

Adapter weights:
* `/content/sft-merchant-agent/final/` or `/kaggle/working/sft-merchant-agent/final/` - Phase A output.
* `/content/grpo-merchant-agent/checkpoint-200/` or `/kaggle/working/grpo-merchant-agent/checkpoint-200/` - Phase B final when `max_steps=200`.

To use a GRPO checkpoint correctly: first merge the SFT adapter into the base, then load the GRPO adapter.
